<a href="https://colab.research.google.com/github/zsh6883-hub/Empire-Quant-Lab/blob/main/2026_06_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import datetime
import time
import random
import warnings
import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import akshare as ak
import google.generativeai as genai

# 全局最高战略防御：阻断一切库级别兼容性冗余输出
warnings.filterwarnings('ignore')

# 核心资产防御集群配置
PORTFOLIO_MAP = {
    "000725": "BOE",
    "600157": "YTP",
    "601991": "DTG",
    "000767": "JDP",
    "000100": "TCL"
}

class EmpireQuantLabTerminalPro:
    """
    城堡量化矩阵控制台 (v3.2.0 滚动演进完全体)
    Evolution Core: 强化二阶动能算法 + 引入 Bias 防追高阈值 + 全链路降级容错
    """
    def __init__(self, initial_cash=1000000.0, max_positions=5, gemini_key="", bias_limit=0.15):
        self.total_cash = initial_cash
        self.available_cash = initial_cash
        self.max_positions = max_positions
        self.position_limit = initial_cash / max_positions
        self.bias_limit = bias_limit  # 偏离度风控红线（默认15%）

        self.portfolio_records = []
        self.audit_logs = {}  # 留存 AI 审计痕迹以备复盘

        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Accept-Language': 'zh-CN,zh;q=0.9',
            'Connection': 'keep-alive'
        }

        self.gemini_key = gemini_key
        genai.configure(api_key=self.gemini_key)
        self.ai_model = genai.GenerativeModel('gemini-1.5-flash')

    # ==========================================
    # 原则一：【先一】 - 链路高抗震流控抓取
    # ==========================================
    def fetch_real_market_data(self, stock_code, lookback_days=120):
        """ 稳固级日线历史行情数据流同步引擎 """
        symbol_prefix = "sh" if stock_code.startswith("60") else "sz"
        full_symbol = f"{symbol_prefix}{stock_code}"

        for attempt in range(3):
            try:
                df_raw = ak.stock_zh_a_daily(symbol=full_symbol, adjust="qfq")
                if df_raw is not None and not df_raw.empty:
                    df = df_raw[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
                    df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
                    df['Date'] = pd.to_datetime(df['Date'])
                    df.set_index('Date', inplace=True)
                    return df.tail(lookback_days)
            except Exception as e:
                if attempt < 2:
                    wait_sec = (attempt + 1) * 3.0
                    print(f"📡 [链路抖动] 标的 {stock_code} 遭遇断连。正在执行第 {attempt+1} 次抗封锁重试，休眠 {wait_sec} 秒...")
                    time.sleep(wait_sec)
                else:
                    print(f"⚠️ [总线断开] {stock_code} 经 3 次冲锋仍被拒绝。启动技术降级，跳过该标的。")
        return None

    # ==========================================
    # 原则二：【再二】 - 均线偏离(Bias)与 MACD 一阶导算法
    # ==========================================
    def calculate_indicators(self, df):
        """ 技术指标矩阵：计算多头排列、MACD及 Bias 乖离度 """
        df = df.copy()
        df['MA5'] = df['Close'].rolling(window=5).mean()
        df['MA20'] = df['Close'].rolling(window=20).mean()

        # 计算 20 日乖离度 $Bias = (Close - MA20) / MA20$
        df['Bias20'] = (df['Close'] - df['MA20']) / df['MA20']

        exp1 = df['Close'].ewm(span=12, adjust=False).mean()
        exp2 = df['Close'].ewm(span=26, adjust=False).mean()
        df['DIF'] = exp1 - exp2
        df['DEA'] = df['DIF'].ewm(span=9, adjust=False).mean()
        df['MACD'] = 2 * (df['DIF'] - df['DEA'])

        # 计算 MACD 柱状图的一阶差分（斜率），判断红柱是在放大还是在缩小
        df['MACD_Slope'] = df['MACD'].diff()
        return df

    def run_ai_fraud_audit(self, ticker, name):
        """ AI 认知内核：执行语义级反欺诈舆情风控 """
        print(f"🔍 [AI 审计节点] 正在拦截 {name} ({ticker}) 的实时异动舆情...")
        api_url = f"https://xueqiu.com/query/v1/symbol/status.json?count=15&comment=0&symbol={ticker}&hl=0&source=all"

        try:
            res = requests.get(api_url, headers=self.headers, timeout=6)
            status_list = res.json().get('list', [])
            combined_sentiment = "".join([s.get('text', '') for s in status_list])

            if not combined_sentiment or len(combined_sentiment.strip()) < 10:
                self.audit_logs[ticker] = "PASSED (Silent Market)"
                return True

            prompt = f"""
            You are an elite Wall Street short-seller forensic accountant auditing {name} ({ticker}).
            Analyze the following recent market discussions and leaks for signs of financial fraud, channel stuffing,
            manipulated contract liabilities, or executive scandals.
            ---
            {combined_sentiment[:1800]}
            ---
            Since our portfolio tolerance is ZERO, if there is even a 1% suspicion of fraud or critical black swan risk,
            you must issue an instant VETO.

            Format your response exactly as:
            CONCLUSION: [PASSED] or [VETOED]
            REASON: (One short English sentence explaining your decision)
            """
            response = self.ai_model.generate_content(prompt)
            result_text = response.text.strip()
            print(f"🧠 [Gemini 审计报告书]:\n{result_text}")

            self.audit_logs[ticker] = result_text
            return "VETOED" not in result_text
        except Exception as e:
            print(f"⚠️ [风控分流] AI 网络波动，通过安全机制实施技术性免检准入。")
            self.audit_logs[ticker] = "PASSED (Network Bypass)"
            return True

    # ==========================================
    # 原则三：【其次三】 - 解耦高阶渲染面板
    # ==========================================
    def render_interactive_dashboard(self, df, ticker, name):
        """ Plotly 赛博朋克暗黑三层交互看板渲染 """
        fig = make_subplots(
            rows=3, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.03,
            row_heights=[0.55, 0.18, 0.27]
        )

        # Subplot 1: 主 K 线与策略均线簇
        fig.add_trace(go.Candlestick(
            x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'],
            name=f'{name} ({ticker}) K-Line'
        ), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['MA5'], name='MA5 Tactical', line=dict(color='#ff9900', width=1.5)), row=1, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['MA20'], name='MA20 Trend', line=dict(color='#00bcff', width=1.5)), row=1, col=1)

        # Subplot 2: 交割量能副图
        v_colors = ['#ff4d4d' if c >= o else '#2ecc71' for c, o in zip(df['Close'], df['Open'])]
        fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='Volume', marker_color=v_colors, opacity=0.85), row=2, col=1)

        # Subplot 3: MACD 共振动能指标
        fig.add_trace(go.Scatter(x=df.index, y=df['DIF'], name='DIF 快线', line=dict(color='#ffffff', width=1.2)), row=3, col=1)
        fig.add_trace(go.Scatter(x=df.index, y=df['DEA'], name='DEA 慢线', line=dict(color='#ffff00', width=1.2)), row=3, col=1)
        m_colors = ['#ff4d4d' if m >= 0 else '#2ecc71' for m in df['MACD']]
        fig.add_trace(go.Bar(x=df.index, y=df['MACD'], name='MACD 柱状图', marker_color=m_colors), row=3, col=1)

        # 样式集约化配置
        fig.update_layout(
            title=dict(text=f"🏰 EMPIRE QUANT v3.2.0 - LIVE TERMINAL FOR {name}", x=0.01, font=dict(color='#ffffff', size=16)),
            template="plotly_dark",
            paper_bgcolor='#0e0e0e',
            plot_bgcolor='#141414',
            xaxis3_rangeslider_visible=False,
            height=650,
            margin=dict(l=40, r=30, t=80, b=40)
        )
        fig.update_yaxes(gridcolor='#222222', zerolinecolor='#333333')
        fig.update_xaxes(gridcolor='#222222')
        fig.show()

    def execute_central_bus(self):
        """ 调度轴心核心运行模块 """
        print("\n" + "🏰 " * 10 + "帝国中央控制总线 v3.2.0 · 矩阵点火" + "🏰 " * 10)

        for code, name in PORTFOLIO_MAP.items():
            if len(self.portfolio_records) >= self.max_positions:
                print("🛡️ [防线就绪] 主力分配仓位已全线饱和，不再开辟新仓位。")
                break

            print(f"\n📡 [1/3 链路建立] 正在接入标的数据流: [{code} | {name}] ...")
            raw_data = self.fetch_real_market_data(stock_code=code, lookback_days=120)

            if raw_data is None or raw_data.empty:
                continue

            processed_df = self.calculate_indicators(raw_data)
            latest_metrics = processed_df.iloc[-1]

            current_close = float(latest_metrics['Close'])
            current_bias = float(latest_metrics['Bias20'])
            current_macd = float(latest_metrics['MACD'])
            macd_slope = float(latest_metrics['MACD_Slope'])

            print(f"📊 [数据透视] {name} 当前价: {current_close:.2f} | 20日乖离度: {current_bias*100:+.2f}% | MACD柱高: {current_macd:.3f}")

            # ==========================================
            # 【算法进化】：二阶多头动能共振筛选器
            # ==========================================
            # 条件 A: 价格必须运行在 20 日趋势线之上
            condition_trend = current_close > latest_metrics['MA20']
            # 条件 B: MACD 处于正向多头区，或者红柱正在加速放大（一阶导数为正）
            condition_momentum = current_macd > 0 or macd_slope > 0

            if condition_trend and condition_momentum:
                # 触发【乖离度防追高拦截】
                if current_bias > self.bias_limit:
                    print(f"🛑 [风控过热拦截] 标的 {name} 偏离 MA20 过远 ({current_bias*100:.1f}%), 触发防追高保护，放弃建仓。")
                    continue

                print(f"🟢 [2/3 技术筛选通过] {name} ({code}) 触发动能共振共鸣。")

                # 调配 AI 认知内核
                ai_clearance = self.run_ai_fraud_audit(ticker=code, name=name)
                if not ai_clearance:
                    print(f"🛑 [AI 一票否决] {name} 舆情存在潜在黑天鹅尾部风险，强制撤单。")
                    continue

                # 资金分配及自动化交割
                shares_to_buy = int((self.position_limit / current_close) // 100) * 100
                cost = shares_to_buy * current_close

                if shares_to_buy > 0 and self.available_cash >= cost:
                    self.available_cash -= cost
                    self.portfolio_records.append({
                        'Ticker': code, 'Asset': name, 'Shares': f"{shares_to_buy:,}",
                        'Entry_Price': current_close, 'Bias': f"{current_bias*100:+.1f}%",
                        'Capital_Allocated': round(cost, 2)
                    })
                    print(f"💰 [3/3 主板资产交割] 完成 {name} 资产吸纳。划转金额: {cost:,.2f} RMB。")

                    # 渲染交互面板
                    self.render_interactive_dashboard(processed_df, ticker=code, name=name)
            else:
                print(f"🔴 [技术拦截] 标的 {name} 动能指标未发生共振，资产不予并网。")

            # 🛡️ 强制阻尼延迟：完美消解海外机房反爬特征
            time.sleep(random.uniform(2.0, 3.5))

        self._print_final_report()

    def _print_final_report(self):
        """ 持仓、风控与 AI 审计溯源总览 """
        print("\n" + "🛡️ " * 12 + " 帝国资产持仓与风控实时总览 " + "🛡️ " * 12)
        if self.portfolio_records:
            print(pd.DataFrame(self.portfolio_records).to_string(index=False))
        else:
            print("⚪ 动态风控升级，集群处于全线空仓现金安全防护垫状态。")

        allocated_funds = sum([p['Capital_Allocated'] for p in self.portfolio_records])
        print("-" * 75)
        print(f"🏦 账户可用安全现金储备  : {self.available_cash:,.2f} RMB")
        print(f"👑 帝国集群实时防御总净值: {self.available_cash + allocated_funds:,.2f} RMB")
        print("🛡️ " * 31 + "\n")

if __name__ == "__main__":
    # ⚠️ 请在这里输入真实的 Gemini API 密匙
    YOUR_REAL_API_KEY = "AIzaSy..."

    # 调动控制总线（初始100万，每只限制20万，偏离度红线限制15%）
    terminal_commander = EmpireQuantLabTerminalPro(
        initial_cash=1000000.0,
        max_positions=5,
        gemini_key=YOUR_REAL_API_KEY,
        bias_limit=0.15
    )

    # 启动全自动量化控制矩阵
    terminal_commander.execute_central_bus()


🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 帝国中央控制总线 v3.2.0 · 矩阵点火🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 🏰 

📡 [1/3 链路建立] 正在接入标的数据流: [000725 | BOE] ...
📊 [数据透视] BOE 当前价: 5.47 | 20日乖离度: +15.74% | MACD柱高: 0.160
🛑 [风控过热拦截] 标的 BOE 偏离 MA20 过远 (15.7%), 触发防追高保护，放弃建仓。

📡 [1/3 链路建立] 正在接入标的数据流: [600157 | YTP] ...
📊 [数据透视] YTP 当前价: 1.84 | 20日乖离度: +3.05% | MACD柱高: 0.019
🟢 [2/3 技术筛选通过] YTP (600157) 触发动能共振共鸣。
🔍 [AI 审计节点] 正在拦截 YTP (600157) 的实时异动舆情...
💰 [3/3 主板资产交割] 完成 YTP 资产吸纳。划转金额: 199,824.00 RMB。



📡 [1/3 链路建立] 正在接入标的数据流: [601991 | DTG] ...
📊 [数据透视] DTG 当前价: 9.18 | 20日乖离度: +26.90% | MACD柱高: 0.178
🛑 [风控过热拦截] 标的 DTG 偏离 MA20 过远 (26.9%), 触发防追高保护，放弃建仓。

📡 [1/3 链路建立] 正在接入标的数据流: [000767 | JDP] ...
📊 [数据透视] JDP 当前价: 6.11 | 20日乖离度: +24.62% | MACD柱高: 0.241
🛑 [风控过热拦截] 标的 JDP 偏离 MA20 过远 (24.6%), 触发防追高保护，放弃建仓。

📡 [1/3 链路建立] 正在接入标的数据流: [000100 | TCL] ...
📊 [数据透视] TCL 当前价: 4.43 | 20日乖离度: +1.58% | MACD柱高: 0.023
🟢 [2/3 技术筛选通过] TCL (000100) 触发动能共振共鸣。
🔍 [AI 审计节点] 正在拦截 TCL (000100) 的实时异动舆情...
💰 [3/3 主板资产交割] 完成 TCL 资产吸纳。划转金额: 199,793.00 RMB。



🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️  帝国资产持仓与风控实时总览 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 
Ticker Asset  Shares  Entry_Price  Bias  Capital_Allocated
600157   YTP 108,600         1.84 +3.1%           199824.0
000100   TCL  45,100         4.43 +1.6%           199793.0
---------------------------------------------------------------------------
🏦 账户可用安全现金储备  : 600,383.00 RMB
👑 帝国集群实时防御总净值: 1,000,000.00 RMB
🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 🛡️ 

